# Lab 02-1 - Hierarchical Clustering (Solutions)

In this lab we will transition towards focusing on machine learning methods themselves and away from the inner working of Python, `numpy`, and `pandas`. You should be prepared to use information from previous labs, but also to use published documentation to learn about useful functions, methods, and attributes on your own.

In [ ]:
## Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

Examples throughout the lab will use the "moons" and "blobs" toy data sets that were introduced in a previous lab:

In [ ]:
## Datasets module
from sklearn import datasets

## Toy Dataset #1 - "moons"
moons = datasets.make_moons(n_samples=500, noise=0.11, random_state=8)
moons_X = moons[0]         # features to use in clustering
moons_labels = moons[1]    # true cluster identities

## Visualization
plt.scatter(moons_X[:,0], moons_X[:,1], c = moons_labels)
plt.show()

## Toy Dataset #2 - "blobs"
blobs = datasets.make_blobs(n_samples=500, cluster_std=2, random_state=8)
blobs_X = blobs[0]         # features to use in clustering
blobs_labels = blobs[1]    # true cluster identities

## Visualization
plt.scatter(blobs_X[:,0], blobs_X[:,1], c = blobs_labels)
plt.show()

## Part 1 - Agglomerative clustering

### Model Fitting and Dendrograms

Agglomerative clustering is implemented in the `cluster` module of `sklearn`. The basic workflow of many models implemented in sklearn is to first initialize a model then fit to your data using its `.fit()` method below. The arguments `distance_threshold=0` and `n_clusters=None` are used to ensure that the full dendrogram is fit to the data.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
agg_cluster_single = AgglomerativeClustering(
    linkage = 'single', 
    distance_threshold = 0, 
    n_clusters = None).fit(moons_X)

Unfortunately there not currently an official function to plot dendrograms in `sklearn`, but the following function, which is [described here](https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html), can be used to create a basic dendrogram. Don't concern yourself too much with understanding how this code works; rather take it as given.

In [ ]:
from scipy.cluster.hierarchy import dendrogram

def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)

In [ ]:
## Part of the dendrogram for "moons"

# The p argument sets how deep down the dendrogram tree should we plot.
# Try different values and see how the numbers in parentheses change
plot_dendrogram(model = agg_cluster_single, truncate_mode="level", p=10) 
plt.xlabel("Index of obs (or # of obs in branch if in parentheses)")
plt.ylabel("Distance between merged clusters")
plt.show()

A few things you should notice in this example:

1. The function relies upon the `dendogram()` function in the `scipy` library. This function is assigning colors to the branches of the dendrogram via its default arguments. You can [read more here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html#scipy.cluster.hierarchy.dendrogram). 
1. Along the x-axis of the dendrogram:
    1. numbers without parentheses are the index positions (row numbers) of invidual observations (row numbers)
    1. numbers appearing in parentheses indicate the number of observations in that branch
1. We used the single linkage criterion, which is what causes the "chaining" behavior that can be seen in this subsection of the dendrogram (notably on the right side of the graph where samples are merged into the cluster one by one)

Let's see what the fitted clusters actually look like. For this, we need to apply the `.fit_predict()` method to apply our cluster model to the `moons_X` data. This will give us the label of the cluster that each observation is assigned to. For example, the first observation is assigned to a cluster labeled `485`.Notice we have a lot of unique values i.e. a lot of unique resulting clusters.

In [ ]:
agg_cluster_single_predicted = AgglomerativeClustering(
    linkage = 'single', 
    distance_threshold = 0, 
    n_clusters = None).fit_predict(moons_X)
agg_cluster_single_predicted

Let's plot these clusters where we set `c` color to be the cluster labels. Again, we see a lot of unique clusters.

In [ ]:
plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cluster_single_predicted) 
plt.show()

**Question 1**:

- **Part A**: Apply agglomerative clustering using ward linkage to the "moons" data set. Display the first 10 levels of the resulting dendrogram.
- **Part B**: How does this dendrogram compare to the example that used single linkage? Highlight any noticeable differences you see and what they say about the nature of the resulting clusters.
- **Part C**: Plot the observations' cluster labels. What do you notice different?

In [ ]:
# Solution code for Q1.A)
agg_cluster_ward = AgglomerativeClustering(
    linkage = 'ward',
    distance_threshold = 0,
    n_clusters = None).fit(moons_X)

plot_dendrogram(model = agg_cluster_ward, truncate_mode="level", p=10)
plt.xlabel("Index of obs (or # of obs in branch if in parentheses)")
plt.ylabel("Distance between merged clusters")
plt.title("Ward linkage")
plt.show()

In [ ]:
# Solution code for Q1.B)
# At p=10 the labels overplot badly. Truncating to p=3 makes the top of the
# tree - which is the part that matters - readable.
plot_dendrogram(model = agg_cluster_ward, truncate_mode="level", p=3, no_labels=True)
plt.xlabel("Index of obs (or # of obs in branch if in parentheses)")
plt.ylabel("Distance between merged clusters")
plt.title("Ward linkage, truncated at p=3")
plt.show()

# The heights of the last few merges tell the story numerically:
print("Last 5 merge distances, ward:  ", np.round(agg_cluster_ward.distances_[-5:], 3))
print("Last 5 merge distances, single:", np.round(agg_cluster_single.distances_[-5:], 3))

**Solution for Q1.B):** Three differences stand out.

1. **Balance.** Recall that numbers in parentheses are the number of observations in a branch. Under ward linkage the branches split into comparably-sized groups (e.g. the top split is roughly 330 vs. 170), whereas under single linkage observations get peeled off one at a time and you see many branches of size `(1)`. That one-at-a-time peeling is the "chaining" behavior mentioned above.

2. **Scale of the y-axis.** Ward's final merges happen at distances of 7.5, 7.6, 8.7, 13.6, and 22.8, while single linkage's final merges all happen at distances of about 0.16 to 0.22. Single linkage measures the distance between the *closest* pair of points in two clusters, and in a dense 500-point cloud there is always some pair that is close, so the tree never "stretches". Ward measures how much within-cluster variance a merge creates, which grows as the clusters get big.

3. **A clear cut point.** Because of that, the ward dendrogram has an obvious place to cut: the jump from 13.6 to 22.8 is much larger than the jumps below it, which suggests $k=2$. The single linkage dendrogram has no such gap, so it gives you no guidance about how many clusters to take.

The practical upshot is that ward produces clusters made of points that are *collectively* similar, while single linkage only requires each point to be near *one* other point in its cluster.

In [ ]:
# Solution code for Q1.C)
agg_cluster_ward_predicted = AgglomerativeClustering(
    linkage = 'ward',
    distance_threshold=0,
    n_clusters=None).fit_predict(moons_X)

plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cluster_ward_predicted)
plt.title("Ward, full tree (distance_threshold=0)")
plt.show()

print("Number of unique ward labels:  ", len(np.unique(agg_cluster_ward_predicted)))
print("Number of unique single labels:", len(np.unique(agg_cluster_single_predicted)))

In [ ]:
# Cutting the two trees to k=2 is what actually reveals the difference:
for linkage_type in ['single', 'ward']:
    labels_k2 = AgglomerativeClustering(
        linkage = linkage_type,
        n_clusters = 2).fit_predict(moons_X)
    plt.scatter(moons_X[:,0], moons_X[:,1], c = labels_k2)
    plt.title(f"{linkage_type} linkage, k=2: cluster sizes {np.bincount(labels_k2)}")
    plt.show()

**Solution for Q1.C):** The honest answer to "what do you notice different?" for the *full* tree is: **nothing**. Because `distance_threshold=0` forces the tree all the way down to its leaves, both models return 500 clusters of one observation each, so both scatter plots are just 500 differently-coloured points. The colors carry no spatial meaning in either plot. This is itself worth internalizing - a fully-grown dendrogram does not hand you usable cluster labels.

The difference appears as soon as you **cut** the trees. At $k=2$:

- **Single linkage** gives cluster sizes of **499 and 1**. It puts essentially every observation in one cluster and isolates a single outlying point. The chaining behavior means there is always a short hop from the main blob to the next-nearest point, so the algorithm never finds a reason to split the data in two.
- **Ward linkage** gives cluster sizes of **330 and 170** and produces a visible left/right split of the two moons.

Neither one recovers the true moon shapes - that is the limitation this whole section is building toward - but ward at least produces two substantial groups, while single linkage produces a degenerate one.

### Cluster Labels

It is possible to "cut" a hierarchical clustering model to obtain a set $k$ non-overlapping clusters.

This practice allows us to compare the performance of hierarchical methods to alternatives like $k$-means, at least when we know the true clusters like we do in the simulated "moons" and "blobs" examples. In real-world applications, we must use our own judgement to decide if a clustering algorithm is producing useful and appropriate results.

The number of clusters can be chosen based upon context of the application, or inspection of the dendrogram to find a location where the height differences between merges begins to become large (an approach analogous to the "elbow" approach in 
$k$-means).

Below we refit our single linkage agglomerative clustering using $k=3$ and display the results. In order to plot the results, we use the `fit_predict()` method to get the predicted cluster labels for the given data:

In [ ]:
agg_cluster_single_3 = AgglomerativeClustering(
    linkage = 'single', 
    n_clusters = 3).fit_predict(moons_X)
plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cluster_single_3) 
plt.show()

In this example we see the failure of the single linkage method. Other linkage criteria, such as 'complete' tend to perform a little better but not great:

In [ ]:
agg_cl_complete_3 = AgglomerativeClustering(
    linkage = 'complete', 
    n_clusters=3).fit_predict(moons_X)
plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cl_complete_3) 
plt.show()

You might also have noticed that we did not standardize our features, so these clusters might be disproportionately influenced by our first feature (shown on the x-axis). Unfortunately, standardization isn't enough to yield the results we desire:

In [ ]:
from sklearn.preprocessing import StandardScaler
moons_XS = StandardScaler().fit_transform(moons_X) 

agg_cl_complete_3 = AgglomerativeClustering(
    linkage = 'complete', 
    n_clusters=3).fit_predict(moons_XS)
plt.scatter(moons_XS[:,0], moons_XS[:,1], c = agg_cl_complete_3) 
plt.show()

**Question 2**:

- **Part A**: Analyze the "blobs" data set using agglomerative clustering to obtain $k=3$ clusters. Use scatter plots and your own judgements to decide upon an acceptable linkage criterion. Display a scatter plot showing cluster label assignments of your choice. Standardize the data before you begin your analysis.
- **Part B**: Analyze the "blobs" data set using $k$-means clustering to obtain $k=3$ clusters. Display a scatter plot showing cluster label assignments of your choice. Standardize the data before you begin your analysis.
- **Part C**: Comment on any differences. Which method to you prefer and why?

In [ ]:
# Solution code for Q2.A)
# Standardize first: the question asks for it, and the two features are on
# different scales, so distances would otherwise be driven by the x-axis.
blobs_XS = StandardScaler().fit_transform(blobs_X)

# Rather than guessing at a linkage criterion, try all four and compare.
# A cluster of size 1 is a red flag: it means the method isolated an outlier
# instead of finding structure.
for linkage_type in ['single', 'average', 'complete', 'ward']:
    labels = AgglomerativeClustering(
        linkage = linkage_type,
        n_clusters=3).fit_predict(blobs_XS)
    plt.scatter(blobs_XS[:,0], blobs_XS[:,1], c = labels)
    plt.title(f"{linkage_type} linkage: cluster sizes {np.bincount(labels)}")
    plt.show()

**Solution for Q2.A):** The four criteria split cleanly into two groups.

- **Single** (sizes 166, 333, 1) and **average** (sizes 333, 166, 1) both fail. Each one merges two of the three true blobs together and then spends its third cluster on a single isolated point.
- **Complete** (sizes 163, 166, 171) and **ward** (sizes 165, 166, 169) both succeed, returning three groups of almost exactly the size we would expect from `make_blobs(n_samples=500)`.

This matches the general guidance about these criteria: single linkage chains and is good for long, irregular shapes but fragile in the presence of noise; complete and ward both favour compact, roughly spherical clusters, which is exactly what the "blobs" data are made of. I will use **ward** going forward, since it had the most even cluster sizes, but complete would be an equally defensible choice.

In [ ]:
# Fit the chosen criterion
agg_ward = AgglomerativeClustering(
    linkage = 'ward',
    n_clusters=3).fit_predict(blobs_XS)

plt.scatter(blobs_XS[:,0], blobs_XS[:,1], c = agg_ward)
plt.title("Ward linkage, k=3")
plt.show()

In [ ]:
# Solution code for Q2.B)
from sklearn.cluster import KMeans # Import k-means function
import warnings
warnings.filterwarnings("ignore") ## We will turn off warnings since k-means will warn about future changes

# k-means with k = 3. We set random_state so the cluster labels are reproducible.
# Note that random.seed() would NOT work here: sklearn uses numpy's random
# number generator, not the one in the random module.
kmeans = KMeans(n_clusters=3, random_state=76)
kmeans_results = kmeans.fit_predict(blobs_XS)

plt.scatter(blobs_XS[:,0], blobs_XS[:,1], c = kmeans_results)
plt.title("k-means, k=3")
plt.show()

In [ ]:
# Solution code for Q2.C)
# Cluster labels are exchangeable, so we cannot compare them directly.
# Cross-tabulating the two sets of labels shows how they line up:
print(pd.crosstab(agg_ward, kmeans_results,
                  rownames=['ward'], colnames=['kmeans']))

# Compare each method against the true blob identities, again by cross-tab
print("\nWard vs truth:")
print(pd.crosstab(blobs_labels, agg_ward, rownames=['true'], colnames=['ward']))
print("\nk-means vs truth:")
print(pd.crosstab(blobs_labels, kmeans_results, rownames=['true'], colnames=['kmeans']))

**Solution for Q2.C):** The two methods give almost exactly the same answer. Reading the ward-vs-k-means table by its largest entry in each row, the two labelings agree on **498 of the 500** points; the only two disagreements are points sitting in the overlap between two blobs, where the "right" answer is genuinely ambiguous.

Checking each against the true blob identities tells the same story: both recover the three blobs almost perfectly, with **ward misplacing 4 points (99.2% correct)** and **$k$-means misplacing 6 (98.8% correct)**. That gap is 2 observations out of 500 and should not be read as evidence that one method is better - rerun `make_blobs` with a different `random_state` and it could easily flip.

**Which do I prefer?** Since accuracy is effectively tied, the decision comes down to practical considerations, and on those I prefer **$k$-means** for these data:

- The blobs are compact and roughly spherical, which is precisely the shape assumption $k$-means makes, so we give up nothing by using it.
- $k$-means scales far better. Agglomerative clustering must compute and store distances between all pairs of points, which becomes impractical as $n$ grows; $k$-means only compares each point to $k$ centroids.
- $k$-means can assign *new* observations to a cluster with `.predict()`. A hierarchical model has no natural way to do this - you would have to refit the entire tree.

The tradeoffs run the other way too: $k$-means requires committing to $k$ up front and gives a different answer depending on its random initialization (hence `random_state`), whereas the dendrogram shows you every $k$ at once and is deterministic. If I did not already know to expect three spherical groups - as in Question 1's "moons", where the clusters are not spherical at all - the dendrogram would be the more informative place to start.